# Polynomial Regression Pipeline - Quick Notes

A **pipeline** connects steps in the right order. Here it first makes polynomial features, then fits Linear Regression.

## Why use a pipeline?

Without a pipeline, you must remember: make $x^2$ → train the model → make $x^2$ for new data → predict.

A pipeline remembers this recipe for you:

`input x  →  PolynomialFeatures  →  LinearRegression  →  prediction`

Like a small assembly line: the first machine creates useful columns; the second machine learns from them.

## Degree in simple words

| Degree | Extra feature columns from one input `x` | Typical shape |
|---|---|---|
| 1 | `x` | straight line |
| 2 | `x`, `x²` | one smooth bend / U-shape |
| 3 | `x`, `x²`, `x³` | more flexible bend |
| very high | many powers | can wiggle too much |

Higher degree is **not** automatically better. A model that tries to touch every training dot may fail on new dots. That is called **overfitting**.

## The reusable function

`make_pipeline(...)` creates the two-step recipe. When we call `fit`, it runs the steps from left to right. When we call `predict`, it again makes the same feature columns before predicting.

We use `include_bias=False` because `LinearRegression` already adds the constant/intercept itself.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

# Make simple U-shaped data. The small noise is like real-life messiness.
rng = np.random.default_rng(7)
X = rng.uniform(-3, 3, size=(70, 1))
y = 0.7 * X[:, 0] ** 2 + 1 + rng.normal(0, 0.55, size=70)

# Keep 20% unseen. We will use it to check whether the model learned a useful rule.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

def polynomial_pipeline(degree):
    """Return a two-step model: create powers first, then fit a line to them."""
    return make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression()
    )

# Example: degree 2 makes [x, x²].
example_features = PolynomialFeatures(degree=2, include_bias=False)
print('For x = 2, degree 2 creates:', example_features.fit_transform([[2]])[0])
print('Pipeline steps:', polynomial_pipeline(2).steps)

## Visual: too simple, useful, too wiggly

The dots are the same in all three pictures. Only the degree changes. Watch how the high-degree curve starts chasing small random bumps.

In [ ]:
degrees = [1, 2, 25]
labels = ['Degree 1: underfit', 'Degree 2: useful fit', 'Degree 25: overfit risk']
colors = ['#d62828', '#2a9d8f', '#6a4c93']
X_draw = np.linspace(-3, 3, 400).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, degree, label, color in zip(axes, degrees, labels, colors):
    model = polynomial_pipeline(degree)
    model.fit(X_train, y_train)  # Pipeline makes features, then learns the regression weights.
    test_r2 = r2_score(y_test, model.predict(X_test))

    ax.scatter(X_train, y_train, s=32, color='#1d3557', label='Train dots', zorder=3)
    ax.scatter(X_test, y_test, s=32, color='#f4a261', label='Test dots', zorder=3)
    ax.plot(X_draw, model.predict(X_draw), color=color, linewidth=2.7,
            label=f'Test R² = {test_r2:.2f}')
    ax.set_title(label, color=color, weight='bold')
    ax.set_xlabel('Input x')
    ax.grid(alpha=0.2)

axes[0].set_ylabel('Answer y')
axes[0].legend(fontsize=8)
plt.suptitle('Choose degree using unseen test/validation data', weight='bold', y=1.03)
plt.tight_layout()
plt.show()

## How to choose the degree

1. Try a few small values: 1, 2, 3, 4...
2. For each degree, check validation or test performance.
3. Choose the **smallest** degree that works well on unseen data.

Do not choose a degree only because its training score is highest. A perfect training score can be a warning sign.

## Quick revision

- A pipeline prevents us from forgetting preprocessing steps.
- `PolynomialFeatures` creates powers such as $x^2$.
- `LinearRegression` learns after those features are created.
- `fit` learns from training data; `predict` follows the same pipeline recipe for new data.
- Use unseen data to avoid choosing an overfitted degree.